# MNIST with a convolutional network (Keras)

> **Keras baseline for comparison.** Unlike the course notebooks, which build every layer in NumPy, this lab notebook uses TensorFlow/Keras. It exists to show where the from-scratch networks stand against a standard framework on a standard benchmark. Install the extra dependencies with `uv sync --group lab`.


## From Vectors to Vision: Introducing Convolutional Neural Networks

Our previous model — a Multi-Layer Perceptron — treated each image as a flat vector of 784 numbers.

But... does that make sense?

When humans look at a digit, we don't scan each pixel one-by-one.  
We recognize *shapes*, *edges*, *patterns*.  
A "9" looks like a loop with a stick. A "3" has two bumps.  
And those patterns live in the **spatial structure** of the image.

A Convolutional Neural Network (CNN) captures that spatial information.  
It doesn't flatten. It slides filters across the image, detecting local patterns like edges, curves, and corners.

Let’s now build a CNN — a model that *sees* like a machine, but learns like us.

In [1]:
from tensorflow.keras.datasets import mnist

(X_train, y_train), (X_test, y_test) = mnist.load_data()

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))

The images are 28×28 arrays of integers in [0, 255]. Two preparation steps before training.

First, as in every notebook of the course, scale the pixels to [0, 1]:


In [2]:
X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

Second, add a channel dimension. A convolution expects `(height, width, channels)`; grayscale images have a single channel.


In [3]:
X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

In [4]:
X_train.shape, X_test.shape

((60000, 28, 28, 1), (10000, 28, 28, 1))

Each image is now a 28×28 slab of thickness 1: 60,000 of them for training, 10,000 for testing.

The labels are integers 0–9. We keep them as integers and use the *sparse* categorical cross-entropy, which is the same loss as one-hot labels with the categorical cross-entropy, without materialising the one-hot matrix.


## The model

Two convolution + max-pooling blocks learn local patterns (edges, then strokes), then a small dense head classifies. The architecture is the one in `train_cnn.py`.


In [5]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

model = keras.Sequential(
    [
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ]
)
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 225,034 (879.04 KB)

 Trainable params: 225,034 (879.04 KB)

 Non-trainable params: 0 (0.00 B)

## Training

Five epochs of Adam with 10% of the training images held out for validation. The test set is touched once, at the end.


In [6]:
history = model.fit(
    X_train, y_train, validation_split=0.1, epochs=5, batch_size=128, verbose=2
)


Epoch 1/5


422/422 - 37s - 89ms/step - accuracy: 0.9390 - loss: 0.2134 - val_accuracy: 0.9848 - val_loss: 0.0583


Epoch 2/5


422/422 - 36s - 85ms/step - accuracy: 0.9816 - loss: 0.0587 - val_accuracy: 0.9877 - val_loss: 0.0436


Epoch 3/5


422/422 - 31s - 74ms/step - accuracy: 0.9876 - loss: 0.0400 - val_accuracy: 0.9890 - val_loss: 0.0376


Epoch 4/5


422/422 - 27s - 64ms/step - accuracy: 0.9908 - loss: 0.0296 - val_accuracy: 0.9900 - val_loss: 0.0348


Epoch 5/5


422/422 - 26s - 62ms/step - accuracy: 0.9927 - loss: 0.0228 - val_accuracy: 0.9900 - val_loss: 0.0362


In [7]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
errors = int(round((1 - test_accuracy) * len(y_test)))
print(f"Test accuracy: {test_accuracy:.4f} ({errors} errors on {len(y_test):,} images)")


Test accuracy: 0.9889 (111 errors on 10,000 images)


## Reading the result

111 errors on the 10,000 test digits (98.9%), against 359 errors (96.4%) for the dense network of the MNIST case study (`lab/mnist/mnist.ipynb`): about a third of the errors. Convolutions share the same small filters across every position of the image, so a stroke learned in one place is recognised everywhere.

That is the step the course's fully connected networks cannot take on the cat/dog photos, where they memorise the training set but stay close to chance on unseen images.
